# AI Shoe Vendor 

You will be given a shoe collection by interacting with the chatbot. You can also purchase your desired shoes from the collection by giving your credit card creditials to the chatbot.

In [ ]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import re

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

In [ ]:
shoe_collection = { "air jordan 1 low" : "$102", 
                   "yeezy boost 350 v2" : "$220", 
                   "puma suede classic" : "$65", 
                   "reebok club c 85" : "$75", 
                   "new balance 990v5" : "$175", 
                   "converse chuck taylor all star" : "$55", 
                   "vans old skool" : "$60", 
                   "asics gel-kayano 27" : "$160",
                     "cloudsurfer max ": "180"}

In [ ]:
shoe_list_formatted = "\n".join([f"- {name.title()}: {price}" for name, price in shoe_collection.items()])

In [ ]:
# Updated system message with actual shoe collection
system_message = f"""You are a shoe vendor that offers customers a collection of shoes.

Available Shoes:
{shoe_list_formatted}

Instructions:
1. Begin by presenting the available shoes with prices and brief descriptions
2. You are ONLY allowed to recommend shoes from the provided collection above
3. Help customers choose based on their preferences
4. Once a customer selects shoes, process their payment
5. Provide a purchase summary including total cost
6. If payment fails, inform the customer and ask them to try again

IMPORTANT: Never suggest or mention any shoes not in the list above."""

In [ ]:
def get_shoe_price(shoe_name):
    print(f"Tool get_shoe_price called for {shoe_name}")
    shoe_name = shoe_name.lower()
    return shoe_collection.get(shoe_name, "Shoe not found")

In [ ]:
def validate_credit_card(card_number):
    """Basic credit card validation using Luhn algorithm"""
    card_number = re.sub(r'\D', '', str(card_number))  # Remove non-digits
    if len(card_number) < 13 or len(card_number) > 19:
        return False
    
    # Luhn algorithm
    def luhn_check(card_num):
        digits = [int(d) for d in card_num]
        for i in range(len(digits) - 2, -1, -2):
            digits[i] *= 2
            if digits[i] > 9:
                digits[i] -= 9
        return sum(digits) % 10 == 0
    
    return luhn_check(card_number)

def validate_expiry_date(expiry):
    """Validate expiry date format MM/YY"""
    pattern = r'^(0[1-9]|1[0-2])\/([0-9]{2})$'
    return bool(re.match(pattern, expiry))

def validate_cvv(cvv):
    """Validate CVV (3-4 digits)"""
    pattern = r'^[0-9]{3,4}$'
    return bool(re.match(pattern, cvv))

In [ ]:
def process_payment(card_number, expiry_date, cvv, cardholder_name, amount):
    print(f"Tool process_payment called for amount {amount}")
    
    # Remove spaces and hyphens from card number
    clean_card_number = re.sub(r'[\s\-]', '', str(card_number))
    
    # Validate input format
    if not validate_credit_card(clean_card_number):
        return {
            "success": False,
            "error": "Invalid credit card number format",
            "transaction_id": None
        }
    
    if not validate_expiry_date(expiry_date):
        return {
            "success": False,
            "error": "Invalid expiry date format. Please use MM/YY format",
            "transaction_id": None
        }
    
    if not validate_cvv(cvv):
        return {
            "success": False,
            "error": "Invalid CVV. Please enter 3-4 digits",
            "transaction_id": None
        }
    
    if not cardholder_name or len(cardholder_name.strip()) < 2:
        return {
            "success": False,
            "error": "Invalid cardholder name",
            "transaction_id": None
        }
    
    # Check for some specific "declined" test cases to simulate real scenarios
    decline_scenarios = [
        ("4000000000000127", "Insufficient funds"),
        ("4000000000000119", "Processing error occurred"), 
        ("4242424242424241", "Card expired"),  # Note: wrong last digit
    ]
    
    for decline_card, error_msg in decline_scenarios:
        if clean_card_number == decline_card:
            return {
                "success": False,
                "error": error_msg,
                "transaction_id": None
            }
    
    # Add a small random chance of payment failure to simulate real-world scenarios
    import random
    if random.random() < 0.05:  # 5% chance of random failure
        random_errors = [
            "Payment processing temporarily unavailable. Please try again.",
            "Card issuer declined the transaction.",
            "Network error occurred. Please try again."
        ]
        return {
            "success": False,
            "error": random.choice(random_errors),
            "transaction_id": None
        }
    
    # If all validations pass, process successful payment
    transaction_id = f"TXN_{random.randint(100000, 999999)}"
    return {
        "success": True,
        "error": None,
        "transaction_id": transaction_id,
        "amount": amount
    }


In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_shoe_price",
    "description": "Get the price of a pair of shoes. Call this whenever you need to know the shoes price, for example when a customer asks 'How much does the Air Jordan 1 Low cost?'.",
    "parameters": {
        "type": "object",
        "properties": {
            "shoe_name": {
                "type": "string",
                "description": "The name of the pair of shoes",
            },
        },
        "required": ["shoe_name"],
        "additionalProperties": False
    }
}

payment_function = {
    "name": "process_payment",
    "description": "Process payment for shoes purchase. Call this when customer provides payment details.",
    "parameters": {
        "type": "object",
        "properties": {
            "card_number": {
                "type": "string",
                "description": "Credit card number",
            },
            "expiry_date": {
                "type": "string",
                "description": "Card expiry date in MM/YY format",
            },
            "cvv": {
                "type": "string",
                "description": "Card CVV/CVC code",
            },
            "cardholder_name": {
                "type": "string",
                "description": "Name on the credit card",
            },
            "amount": {
                "type": "string",
                "description": "Amount to charge (e.g., '$102')",
            }
        },
        "required": ["card_number", "expiry_date", "cvv", "cardholder_name", "amount"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": price_function},
         {"type": "function", "function": payment_function}]

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_response  = handle_tool_call(message)
        messages.append(message)
        messages.append(tool_response)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    
    if function_name == "get_shoe_price":
        shoe = arguments.get('shoe_name')
        price = get_shoe_price(shoe)
        response_content = json.dumps({"shoe_name": shoe, "price": price})
        
    elif function_name == "process_payment":
        card_num = arguments.get('card_number')
        expiry = arguments.get('expiry_date')
        cvv = arguments.get('cvv')
        cardholder =arguments.get('cardholder_name')
        amount = arguments.get('amount')
    
       # Call the actual payment processing function
        payment_result = process_payment(card_num, expiry, cvv, cardholder, amount)
        response_content = json.dumps(payment_result)
    
     
    else:
        response_content = json.dumps({"error": "Unknown function"})
    
    response = {
    "role": "tool",
    "content": response_content,
    "tool_call_id": tool_call.id
        }
    
    return response

In [ ]:
print("\n=== AI SHOE VENDOR WITH PAYMENT SYSTEM ===")
print("Payment Testing Info:")
print("• Most valid credit card formats will work!")
print("• Try: 4111 1111 1111 1111, 4000 0000 0000 0002, 5555 5555 5555 4444")
print("• Use any realistic expiry (MM/YY) and 3-4 digit CVV")
print("• Some test cards will be declined for simulation:")
print("  - 4000 0000 0000 0127 (Insufficient funds)")
print("  - 4000 0000 0000 0119 (Processing error)")
print("  - 4242 4242 4242 4241 (Card expired)")
print("• 5% random chance of payment failure to simulate real scenarios")
print("==========================================\n")
    
   
gr.ChatInterface(fn=chat, type="messages").launch(inbrowser=True)